In [566]:
import pandas as pd

In [567]:
all_stars_df = pd.read_csv('../data/challenge_210_all_stars.csv')

all_stars_df.head()

,Year,Player,Pos,HT,WT,Team,Selection Type,NBA Draft Status,Nationality
0,2016,Stephen Curry,G,6-3,190,Golden State Warriors,Western All-Star Fan Vote Selection,2009 Rnd 1 Pick 7,United States
1,2016,James Harden,SG,6-5,220,Houston Rockets,Western All-Star Fan Vote Selection,2009 Rnd 1 Pick 3,United States
2,2016,Kevin Durant,SF,6-9,240,Golden State Warriors,Western All-Star Fan Vote Selection,2007 Rnd 1 Pick 2,United States
3,2016,Kawhi Leonard,F,6-7,230,San Antonio Spurs,Western All-Star Fan Vote Selection,2011 Rnd 1 Pick 15,United States
4,2016,Anthony Davis,PF,6-11,253,New Orleans Pelicans,Western All-Star Fan Vote Selection,2012 Rnd 1 Pick 1,United States


In [568]:
draft_df = pd.read_csv('../data/challenge_210_draft_info.csv')

draft_df.head()

,Year,Rd1,Rd2,Rd3,Rd4,Rd5,Rd6,Rd7,Rd8,Rd9,Rd10
0,1984,24,23,23.0,23.0,23.0,23.0,23.0,22.0,22.0,22.0
1,1985,24,23,23.0,23.0,23.0,23.0,23.0,NaN,NaN,NaN
2,1986,24,23,23.0,23.0,23.0,23.0,23.0,NaN,NaN,NaN
3,1987,23,23,23.0,23.0,22.0,24.0,23.0,NaN,NaN,NaN
4,1988,25,25,25.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [569]:
# unpivot draft df to allow for join by year and round number

draft_df = pd.melt(draft_df, id_vars=['Year']).rename(columns={'variable': 'round', 'value': 'num picks'})
draft_df['round'] = draft_df['round'].str.replace('Rd', '').astype(int)
draft_df = draft_df.sort_values(by=['Year', 'round'], ascending=[True, True]).reset_index(drop=True)

draft_df.head()

,Year,round,num picks
0,1984,1,24.0
1,1984,2,23.0
2,1984,3,23.0
3,1984,4,23.0
4,1984,5,23.0


In [570]:
# calculate the number of picks that have taken place before each round

draft_df['picks before round'] = 0

for i in range(1, len(draft_df)):
    if draft_df.loc[i, 'Year'] == draft_df.loc[i-1, 'Year']:
        draft_df.loc[i, 'picks before round'] = draft_df.loc[i-1, 'picks before round'] + draft_df.loc[i-1, 'num picks']

draft_df.head()

,Year,round,num picks,picks before round
0,1984,1,24.0,0.0
1,1984,2,23.0,24.0
2,1984,3,23.0,47.0
3,1984,4,23.0,70.0
4,1984,5,23.0,93.0


In [571]:
# parse out draft info from all_stars_df

all_stars_df['draft year'] = all_stars_df['NBA Draft Status'].str[:4].astype(int)
all_stars_df['round'] = all_stars_df['NBA Draft Status'].str.extract('Rnd\\s([0-9]+?)\\s').dropna().astype(int)
all_stars_df['draft round pick'] = all_stars_df['NBA Draft Status'].str.extract('Pick\\s([0-9]+)').dropna().astype(int)

all_stars_df.head()

,Year,Player,Pos,HT,WT,Team,Selection Type,NBA Draft Status,Nationality,draft year,round,draft round pick
0,2016,Stephen Curry,G,6-3,190,Golden State Warriors,Western All-Star Fan Vote Selection,2009 Rnd 1 Pick 7,United States,2009,1.0,7.0
1,2016,James Harden,SG,6-5,220,Houston Rockets,Western All-Star Fan Vote Selection,2009 Rnd 1 Pick 3,United States,2009,1.0,3.0
2,2016,Kevin Durant,SF,6-9,240,Golden State Warriors,Western All-Star Fan Vote Selection,2007 Rnd 1 Pick 2,United States,2007,1.0,2.0
3,2016,Kawhi Leonard,F,6-7,230,San Antonio Spurs,Western All-Star Fan Vote Selection,2011 Rnd 1 Pick 15,United States,2011,1.0,15.0
4,2016,Anthony Davis,PF,6-11,253,New Orleans Pelicans,Western All-Star Fan Vote Selection,2012 Rnd 1 Pick 1,United States,2012,1.0,1.0


In [572]:
# join draft data with all star data

df = pd.merge(draft_df, all_stars_df, how='inner', left_on=['Year', 'round'], right_on=['draft year', 'round'])

df.head()

df[df['Player'] == 'Anthony Mason']

,Year_x,round,num picks,picks before round,Year_y,Player,Pos,HT,WT,Team,Selection Type,NBA Draft Status,Nationality,draft year,draft round pick
5,1988,3,25.0,50.0,2000,Anthony Mason,PF,6-7,250,Miami Heat,Eastern All-Star Coaches Selection,1988 Rnd 3 Pick 3,United States,1988,3.0


In [573]:
# calculate the overall pick number for each player. dedeupe df and select top three

df['overall pick'] = df['picks before round'] + df['draft round pick']

df = df.groupby(['Player', 'NBA Draft Status'])['overall pick'].max().reset_index()[['Player', 'overall pick', 'NBA Draft Status']]

df[['Player', 'overall pick', 'NBA Draft Status']].nlargest(3, 'overall pick')

,Player,overall pick,NBA Draft Status
51,Isaiah Thomas,60.0,2011 Rnd 2 Pick 30
82,Manu Ginobili,57.0,1999 Rnd 2 Pick 28
11,Anthony Mason,53.0,1988 Rnd 3 Pick 3
